# Inference on a single image

Point at one TIFF (or PNG) and a checkpoint. Nothing else is needed — no mask,
no dataset folder, nothing written to disk. Classes are labelled by id.

In [ ]:
# ============ 1. SETTINGS ============
IMAGE_PATH = "../data/plant_test/D_1_0480.tif"    # the one image to run on
CKPT_PATH  = "/pscratch/sd/x/xchong/3RSE/832/combine/out_combined/vitl16-eomt-cityscapes/combined_experiment/checkpoints/best.ckpt"

ALPHA = 0.8      # overlay opacity


In [ ]:
import os

os.environ["LIGHTLY_TRAIN_CACHE_DIR"] = ".cache"
os.environ["LIGHTLY_TRAIN_MODEL_CACHE_DIR"] = ".cache"
os.environ["TORCH_HOME"] = ".cache"

import colorsys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import lightly_train

Image.MAX_IMAGE_PIXELS = None
IGNORE = 255                  # ignore / unannotated; drawn transparent
IGNORE_COLOR = (255, 0, 255)


def auto_color(n):
    """A distinct color per class id, for any number of classes."""
    if n < 20:
        return [int(c * 255) for c in plt.get_cmap("tab20")(n)[:3]]
    r, g, b = colorsys.hsv_to_rgb((n * 0.618033988749895) % 1.0, 0.65, 0.95)
    return [int(r * 255), int(g * 255), int(b * 255)]


# Indexed by class id, so a class keeps the same color across runs.
PALETTE = np.array([auto_color(i) for i in range(256)], dtype=np.uint8)
PALETTE[IGNORE] = IGNORE_COLOR

model = lightly_train.load_model_from_checkpoint(CKPT_PATH)

print(f"image      : {IMAGE_PATH}")
print(f"checkpoint : {CKPT_PATH}")

In [ ]:
# ============ 2. LOAD THE IMAGE, RUN INFERENCE, PLOT ============
def to_uint8_rgb(path, lo=2, hi=98):
    """Read any image as 8-bit RGB for display.

    Tomography TIFFs are often 16-bit or float, so the 2-98 percentile stretch
    is what makes a faint slice visible.
    """
    a = np.asarray(Image.open(path)).astype(np.float64)
    if a.ndim == 3:
        a = a[..., :3].mean(axis=2)                 # collapse to grayscale
    p, q = np.percentile(a, lo), np.percentile(a, hi)
    a = (np.clip(a, p, q) - p) / (q - p) * 255 if q > p else np.zeros_like(a)
    return np.dstack([a.astype(np.uint8)] * 3)


img_path = Path(IMAGE_PATH)
image = to_uint8_rgb(img_path)          # for display only

pred = model.predict(str(img_path))     # the model reads the file itself
pred = pred.detach().cpu().numpy() if hasattr(pred, "detach") else np.asarray(pred)
pred = pred.astype(np.int64)

# image | predicted mask | prediction over the image
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.5))
axes[0].imshow(image)
axes[0].set_title("image")

axes[1].imshow(PALETTE[pred], interpolation="nearest")
axes[1].set_title("predicted mask")

rgba = np.dstack([PALETTE[pred], np.full(pred.shape, int(255 * ALPHA), np.uint8)])
rgba[..., 3][pred == IGNORE] = 0          # ignore pixels stay transparent
axes[2].imshow(image)
axes[2].imshow(rgba, interpolation="nearest")
axes[2].set_title(f"overlay (alpha={ALPHA})")

for ax in axes:
    ax.axis("off")

ids = sorted(np.unique(pred).tolist())
axes[2].legend(
    handles=[mpatches.Patch(color=PALETTE[i] / 255, label=f"class {i}") for i in ids],
    loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8,
)

fig.suptitle(img_path.name, fontsize=11)
fig.tight_layout()
plt.show()

print(f"predicted class ids: {ids}")